In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("IMDB Dataset.csv")

In [8]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [9]:
df.shape

(50000, 2)

In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
df.drop_duplicates(inplace=True)  #without inplace=True we would have to reassign it back to a variable to save changes

In [12]:
df.shape

(49582, 2)

# Pre-processing

### 1. Converting to lowercase

In [13]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [14]:
import re     # regular expression

# str = "abc is the word, abc"
# new_str = re.sub("abc","xyz", str)   # substitute
# print(new_str)

In [15]:
def remove_urls(text):
    text = re.sub(r"http\S+" ,"",text)  # (pattern, repl, string)  eg:- https://www.google.com
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing punctuations

In [16]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)     # any value not in A_Z a-z 0-1 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [17]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the StopWords

In [1]:
import nltk

nltk.download("punkt")    # for tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\aman
[nltk_data]     upadhyay\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to C:\Users\aman
[nltk_data]     upadhyay\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to C:\Users\aman
[nltk_data]     upadhyay\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [18]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [19]:
# sample_text = "I love coding in python!"
# tokens = word_tokenize(sample_text)
# print(tokens)

In [20]:
def remove_stopwords(text):
    tokens = word_tokenize(text)   # tokenize
    stop_words = stopwords.words("english")   # english language all stopwords

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [22]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [24]:
# running -> run
# played -> play
# Porter Stemming

from nltk.stem import PorterStemmer

In [25]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [26]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [27]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [28]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [29]:
df["review"]

0        e revew nted wtchg 1 oz epod ll hook y rght ex...
1        wder ltle producti br br film techniqu unssum ...
2        thought th wder wy spend tme o hot summer week...
3        bsclli re fmli lttle boy jke thk re zomb close...
4        petter mtte love time mey vulli stunng film wt...
                               ...                        
49995    thought th move dd rght good job t wsnt s cret...
49996    bd plot bd dlogu bd ctng dotc drectng nnoyng p...
49997    ctholc tught n prochl elentri school nun tught...
49998    im gog dgree previou comnt side mlt e secd rte...
49999    one expect str trek move hgh rt fn expect move...
Name: review, Length: 49582, dtype: object

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer( max_features=5000 )

X = tf.fit_transform(df["review"])

In [31]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Dataset & Data Loaders

In [32]:
# train test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [33]:
X_train.shape

(39665, 5000)

In [34]:
X_test.shape

(9917, 5000)

In [40]:
y_train.shape

(39665,)

In [36]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()  # PyTorch tensors cannot be directly created from sparse matrices without first converting them into standard, dense arrays.
X_test = X_test.toarray()

In [41]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),    # PyTorch models cannot read NumPy arrays or Pandas DataFrames directly, They speak exclusively in PyTorch Tensors. from_numpy is faster and consumes less memory than torch.tensor() 
    torch.from_numpy(y_train.values).float(),  # 
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [42]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN 

In [45]:
import torch.nn as nn
import torch.optim as optim

In [46]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size= hidden_size
        self.num_layers= num_layers

        # RNN layer
        self.rnn = nn.RNN( input_size, hidden_size, num_layers, batch_first=True ) 

        # fully connected layer
        self.fc = nn.Linear( hidden_size, 1 )

    def forward(self, x):
        # optional => shape(num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps = (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [48]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()   # Binary cross entropy loss
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [49]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb)   # compute loss
        loss.backward() # backprop
        optimizer.step()  # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/10 and loss = 0.18790428340435028
epoch = 2/10 and loss = 0.49279657006263733
epoch = 3/10 and loss = 0.2577842175960541
epoch = 4/10 and loss = 0.28554078936576843
epoch = 5/10 and loss = 0.2660273015499115
epoch = 6/10 and loss = 0.2953188419342041
epoch = 7/10 and loss = 0.2857923209667206
epoch = 8/10 and loss = 0.3263281285762787
epoch = 9/10 and loss = 0.32021066546440125
epoch = 10/10 and loss = 0.2234359234571457


In [53]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        
        tot_vals += yb.size(0)
        correct_vals += ( predicted == yb ).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.46939598668952
